In [ ]:
import numpy as np
import tensorflow as tf

class MNIST_Dataset:
    def __init__(self,):
        self.dataset = tf.keras.datasets.mnist.load_data()
        ((train_values, train_labels), (test_values, test_labels)) = self.dataset


        train_values = (train_values.astype('float32')/255.0).np.reshape(-1, 28 *28)
        test_values = test_values.astype('float32')/255.0.np.reshape(-1, 28*28)
        self.train = (train_values, train_labels)
        self.test = (test_values, test_labels)

    def get_train_data(self):
        """
        returns train data as a tuple with first Value as normalized picture value from mnist
        and second labels
        """
        return self.train

    def get_test_data(self):
        """
        returns train data as a tuple with first Value as normalized picture value from mnist
        and second labels
        """
        return self.test

initilisation of dataset

In [ ]:
class NeuralNet:
    def __init__(self, input_size = 784, hidden_size = 28, output_size = 10):

        # weights and bias layer one
        self.W1 = np.random.randn(input_size, hidden_size) *np.sqrt(2.0/ input_size)
        self.B1 = np.zeros(1, hidden_size)

        # weights and bias layer one
        self.W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2.0/ hidden_size)
        self.B2 = np.zeros(1, output_size)

        # Layer 1
        self.h1 = None
        self.out1 = None

        #layer 2
        self.h2 = None
        self.out2 = None

        self.learning_rate = 0.01

    def relu(self, layer):
        return np.maximum(0, layer)

    def relu_deriv(self, layer):
        return (layer > 0).astype(float)

    def softmax(self, layer):
        exp_layer = np.exp(layer- np.max(layer, axis=1, keepdims=True))
        return exp_layer/ np.sum(exp_layer,axis=1,keepdims=True)

    def foward_prop(self,input_1):
        self.h1 = (input_1 @ self.W1) + self.B1
        self.out1 = self.relu(self.d1)

        self.h2 = (self.h1 @ self.W2) + self.B2
        self.out2 = self.softmax(self.h2)

        return self.out2

    def loss(self, prediction, truth):
        m = truth.shape[0]  # num sample

        one_hot = np.eye(10)[truth] # reshaping label to matrix

        loss = -np.sum(one_hot * np.log(prediction+ 1e-8)) /m # loss funktion 1e-8 prventing -infinety in log

        return loss


    def backprob(self, input_l, prediction, truth):
        m = input.shape[0]
        one_hot = np.eye(10)[truth]

        #output layer

        delta_a2 = prediction - one_hot
        delta_W2 = (self.out1.T @ delta_a2) / m # .T transposition of matrix
        delta_B2 = np.sum(delta_a2,axis=0, keepdims=0)

        #hidden layer

        delta_a1 = delta_a2 @ self.W2.T
        loss_grad = delta_a1 * self.relu_deriv(self.h1)
        delta_W1 = (input_l.T @ loss_grad) / m
        delta_B1 = np.sum(loss_grad, axis=0, keepdims= True) / m

        #Update weights

        self.W2 -= self.learning_rate * delta_W2
        self.B2 -= self.learning_rate * delta_B2
        self.W1 -= self.learning_rate * delta_W1
        self.B1 -= self.learning_rate * delta_B1

    def train_step(self, input, label):
        pred = self.foward_prop(input)
        loss = self.loss(pred, label)
        self.backprob(input, pred, label)
        return loss


    def predict(self, input):
        pred = self.foward(input)
        return np.argmax(pred, axis= 1)


Initilasation of neural net class without functionality yet, using Xavier initilasation 

np.sqrt(2/input) 2 because we use relue wich sets negative values to 0 wich could be 50% of input neurons in ordor to medigate that we set it to 2 so that the desortion on aktivation stays in variation 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score
import itertools

class PlotGraph:
    def __init__(self,image, label):
        plt.figure(figsize=(6,6))
        sns.set_style("whitegrid")


    @staticmethod
    def show_single(image, true_label, pred_label = None, prob = None):
        plt.figure(figsize=(6,6))
        plt.imshow(image.reshape(28,28), cmap='gray')
        title = f"True: {true_label}"
        color = 'black'
        if pred_label is not None:
            correct = true_label == pred_label
            color = 'green' if correct else 'red'
            title += f"     | Prediction = {pred_label}"
            if prob is not None:
                title += f"{prob:.1f}%"
            plt.title(title, color= color, frontsize= 16, pad= 20)
            plt.axis('off')
            plt.tight_layout()
            plt.show()

    @staticmethod
    def show_grid_pred(images, true_labels, pred_labels, cols=5, rows=3, only_errors = False):
        num_pics = cols * rows
        if only_errors:
            errors_idx = np.where(true_labels != pred_labels)[0]
            if len(errors_idx)  == 0:
                print("No errors to show")
                return
            idx = errors_idx[:num_pics]
            title = f"Error Classifications({len(errors_idx)})"
        else:
            idx = np.random.choice(len(images), num_pics, replace = False)
            title = "Random prediction"

        plt.figure(figsize=(12,8))
        for i,idx_i in enumerate(idx):
            plt.subplot(rows, cols, i+1)
            plt.imshow(images[idx_i].reshape(28,28), cmap = 'gray')
            correct = true_labels[idx_i]
            pred = pred_labels[idx_i]
            color = 'green' if correct == pred else 'red'
            plt.title(f"True: {correct} to prediction: {pred}",color= color, frontsize= 12)
            plt.axis('off')
        plt.subtitle(title, frontsize = 16, y= 0.95)
        plt.tight_layout()
        plt.show()

    @staticmethod
    def plot_confusion(true_labels, pred_labels,normalize = False):
        cm = confusion_matrix(true_labels, pred_labels)
        if normalize:
            cm = cm.astype('float')/ cm.sum(axis =1)[:,np.newaxis]
            fmt = '.2f'
            title = 'Confusion Matrix( normalized)'
        else:
            fmt = 'd'
            title = 'Confusion Matrix'

        plt.figure(figsize=(10,8))
        sns.heatmap(cm, annot=True, fmt=fmt, cmap='Blues',xticklabels=range(10), yticklabels= range(10))
        plt.title(title, frontsize= 16)
        plt.ylabel('true label')
        plt.xlabel('Prediction Label')
        plt.tight_layout()
        plt.show()

    @staticmethod
    def plot_training_history(losses, accuracies = None):
        epochs = range(1, len(losses) +1)

        plt.figure(figsize=(12,5))

        plt.subplot(1,2,1)
        plt.plot(epochs, losses, 'b-o', label = 'loss')
        plt.title('Training Loss')
        plt.xlabel('Epoch')
        plt.ylabel('loss')
        plt.legend()
        plt.grid(True)

        if accuracies:
            plt.subplot(1,2,2)
            plt.plot(epochs,accuracies, 'g-o', label = 'Accuracy')
            plt.title('Training Accuaracy')
            plt.xlabel('Epoch')
            plt.ylabel('Accuracy')
            plt.legend()
            plt.grid()

        plt.tight_layout()
        plt.show()

    @staticmethod
    def plot_probailities(probabilities, true_label, top_k=5):
        probs = probabilities[0]
        pred_label = np.argmax(probs)

        top_idx = np.argsort(probs)[-top_k:][::-1]
        top_probs = probs[top_idx]
        top_labels = top_idx

        plt.figure(figsize=(8,5))
        bars = plt.bar(range(top_k), top_probs, color='skyblue', edgecolor= 'black')

        if true_label in top_labels:
            idx = list(top_labels).index(true_label)
            bars[idx].set_color('green')
        if pred_label in top_labels:
            idx = list(top_labels).index(pred_label)
            bars[idx].set_edgecolor('red')
            bars[idx].set_linewidth(2)

        plt.xticks(range(top_k), top_labels)
        plt.ylabel('Probabilities')
        plt.title(f'Prediction: {pred_label}    | Truth: {true_label}')
        plt.ylim(0,1)

        for i,v in enumerate(top_probs):
            plt.text(i,v+0.02, f'{v:.3f}', ha= 'center')

        plt.tight_layout()
        plt.show()